# **Predicting Market Direction with `sklearn.LogisticRegression`**

**Importing Libraries, modules & data**

In [8]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import yfinance as yf
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier
from sklearn.metrics import accuracy_score

In [13]:
companies = ["BSE.NS", "BEL.NS", "HINDZINC.NS", "HINDCOPPER.NS", "HAL.NS"]
data = yf.download(companies, period="1mo", interval="5m")
df = pd.DataFrame(data["Close"])
for company in companies:
    df[f'LR_{company}'] = np.log(df[company] / df[company].shift(1))
    df[f'Dir_{company}'] = np.sign(df[f'LR_{company}'])
# Create lag features with proper naming convention
lag_features = {}
for company in companies:
    lags = []
    for lag in range(1, 6):
        col_name = f'lag{lag}_{company}'
        df[col_name] = df[f'LR_{company}'].shift(lag)
        lags.append(col_name)
    lag_features[company] = lags
# Drop all NAs at once
initial_shape = df.shape
df.dropna(inplace=True)

# Standardize and train
results = {}
for company in companies:
    # Get features for this company
    features = lag_features[company]
    target = f'Dir_{company}'
    feature_means = df[features].mean()
    feature_stds = df[features].std()
    # Standardize
    X = (df[features] - feature_means) / feature_stds
    y = df[target]
    # Train model
    model = OneVsRestClassifier(LogisticRegression(C=1e6, max_iter=10000))
    model.fit(X, y)
    predictions = model.predict(X)
    # Calculate performance
    hits = (predictions == y).sum()
    total = len(y)
    accuracy = hits / total
    # Store results
    results[company] = {
        'model': model,
        'accuracy': accuracy,
        'means': feature_means,
        'stds': feature_stds
    }
    print(f"{company}: Accuracy = {accuracy:.4f}")
df
# Keep standardized features in dataframe for later use
# for company in companies:
#     features = lag_features[company]
#     means = results[company]['means']
#     stds = results[company]['stds']
#     df[[f'std_{f}' for f in features]] = (df[features] - means) / stds

/tmp/ipython-input-72745778.py:2: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(companies, period="1mo", interval="5m")
[*********************100%***********************]  5 of 5 completed


BSE.NS: Accuracy = 0.5220
BEL.NS: Accuracy = 0.5016
HINDZINC.NS: Accuracy = 0.5093
HINDCOPPER.NS: Accuracy = 0.5016
HAL.NS: Accuracy = 0.5144


Ticker,BEL.NS,BSE.NS,HAL.NS,HINDCOPPER.NS,HINDZINC.NS,LR_BSE.NS,Dir_BSE.NS,LR_BEL.NS,Dir_BEL.NS,LR_HINDZINC.NS,...,lag1_HINDCOPPER.NS,lag2_HINDCOPPER.NS,lag3_HINDCOPPER.NS,lag4_HINDCOPPER.NS,lag5_HINDCOPPER.NS,lag1_HAL.NS,lag2_HAL.NS,lag3_HAL.NS,lag4_HAL.NS,lag5_HAL.NS
Datetime,,,,,,,,,,,,,,,,,,,,,
2025-12-24 04:15:00+00:00,401.100006,2721.600098,4429.399902,424.799988,628.099976,-0.001028,-1.0,-0.000249,-1.0,-0.004448,...,-0.002908,0.011801,0.015040,-0.002145,0.011737,-0.001420,-0.000203,-0.000945,0.002681,0.000699
2025-12-24 04:20:00+00:00,401.500000,2732.000000,4431.899902,424.850006,628.200012,0.003814,1.0,0.000997,1.0,0.000159,...,-0.010305,-0.002908,0.011801,0.015040,-0.002145,-0.000993,-0.001420,-0.000203,-0.000945,0.002681
2025-12-24 04:25:00+00:00,401.649994,2734.699951,4429.299805,426.399994,627.549988,0.000988,1.0,0.000374,1.0,-0.001035,...,0.000118,-0.010305,-0.002908,0.011801,0.015040,0.000564,-0.000993,-0.001420,-0.000203,-0.000945
2025-12-24 04:30:00+00:00,401.350006,2734.699951,4425.799805,427.700012,628.349976,0.000000,0.0,-0.000747,-1.0,0.001274,...,0.003642,0.000118,-0.010305,-0.002908,0.011801,-0.000587,0.000564,-0.000993,-0.001420,-0.000203
2025-12-24 04:35:00+00:00,401.450012,2737.199951,4424.799805,426.799988,627.900024,0.000914,1.0,0.000249,1.0,-0.000716,...,0.003044,0.003642,0.000118,-0.010305,-0.002908,-0.000791,-0.000587,0.000564,-0.000993,-0.001420
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-01-23 09:35:00+00:00,408.950012,2685.899902,4300.100098,534.650024,698.700012,-0.000707,-1.0,-0.000855,-1.0,-0.000286,...,-0.001863,-0.002138,0.001580,-0.001673,-0.000835,-0.003849,0.002804,0.001859,-0.001534,-0.000116
2026-01-23 09:40:00+00:00,408.750000,2687.500000,4301.700195,535.000000,700.000000,0.000596,1.0,-0.000489,-1.0,0.001859,...,-0.003081,-0.001863,-0.002138,0.001580,-0.001673,-0.000907,-0.003849,0.002804,0.001859,-0.001534
2026-01-23 09:45:00+00:00,410.250000,2684.000000,4301.399902,534.000000,698.400024,-0.001303,-1.0,0.003663,1.0,-0.002288,...,0.000654,-0.003081,-0.001863,-0.002138,0.001580,0.000372,-0.000907,-0.003849,0.002804,0.001859


In [10]:
Direction

,Dir BSE.NS,Dir BEL.NS,Dir HINDZINC.NS,Dir HINDCOPPER.NS,Dir HAL.NS
Datetime,,,,,
2025-12-24 03:45:00+00:00,NaN,NaN,NaN,NaN,NaN
2025-12-24 03:50:00+00:00,1.0,NaN,NaN,NaN,NaN
2025-12-24 03:55:00+00:00,-1.0,NaN,NaN,NaN,NaN
2025-12-24 04:00:00+00:00,-1.0,NaN,NaN,NaN,NaN
2025-12-24 04:05:00+00:00,-1.0,NaN,NaN,NaN,NaN
...,...,...,...,...,...
2026-01-23 09:35:00+00:00,-1.0,-1.0,-1.0,-1.0,-1.0
2026-01-23 09:40:00+00:00,1.0,-1.0,1.0,1.0,1.0
2026-01-23 09:45:00+00:00,-1.0,1.0,-1.0,-1.0,-1.0


**Scaling & Standardizing Features**
> Scale all Lag in way that mean become 0 and std become 1

In [11]:
a={"class":[[1,2,3],[4,5,6],[45,74],[85,36,741,966,198]],
   "division":[[56,63],[45,62],[45,74],[85,36,741,966,198]],
   "studetns":[[12,15,18],[25,22,30],[45,74],[85,36,741,966,198]],
   "afsd":[[12,15,18],[25,22,30],[85,41,360],[42,12,36]],
   "asds":[[12,15,18],[25,22,30],[85,41,360],[42,12,36]],
   "dgre":[[12,15,18],[25,22,30],[85,41,360],[42,12,36]],
   "etrf":[[12,15,18],[25,22,30],[85,41,360],[42,12,36]]}
b=pd.DataFrame(a,columns=[i for i in a])
b

,class,division,studetns,afsd,asds,dgre,etrf
0,"[1, 2, 3]","[56, 63]","[12, 15, 18]","[12, 15, 18]","[12, 15, 18]","[12, 15, 18]","[12, 15, 18]"
1,"[4, 5, 6]","[45, 62]","[25, 22, 30]","[25, 22, 30]","[25, 22, 30]","[25, 22, 30]","[25, 22, 30]"
2,"[45, 74]","[45, 74]","[45, 74]","[85, 41, 360]","[85, 41, 360]","[85, 41, 360]","[85, 41, 360]"
3,"[85, 36, 741, 966, 198]","[85, 36, 741, 966, 198]","[85, 36, 741, 966, 198]","[42, 12, 36]","[42, 12, 36]","[42, 12, 36]","[42, 12, 36]"
